# Clase 049 — async / httpx / aiohttp para data scientists

Self-contained: levantamos un **mock server local con aiohttp** (sin internet) y comparamos `requests` sync vs `httpx.AsyncClient` con `asyncio.gather`. Vemos Semaphore, retry con backoff y patrón productor-consumidor.

In [ ]:
import asyncio, time, threading, random

try:
    import httpx
    import aiohttp
    from aiohttp import web
    import requests
except ImportError as e:
    raise ImportError('Instalá: pip install httpx aiohttp requests') from e

# En Jupyter el loop ya corre — habilitamos nesting si está disponible
try:
    import nest_asyncio; nest_asyncio.apply()
except ImportError:
    pass

random.seed(42)
print(f'httpx {httpx.__version__} | aiohttp {aiohttp.__version__} | requests {requests.__version__}')

## 1. Mock server local (aiohttp en un thread aparte)

Cada request espera `?delay=0.1` segundos antes de responder. Simula API lenta sin depender de httpbin.

In [ ]:
HOST, PORT = '127.0.0.1', 8765
BASE = f'http://{HOST}:{PORT}'
_server_loop = None
_server_runner = None

async def handle_delay(request):
    delay = float(request.query.get('delay', '0.1'))
    # 10% de las requests devuelven 503 para probar retry
    if random.random() < 0.10:
        return web.Response(status=503, text='busy')
    await asyncio.sleep(delay)
    return web.json_response({'id': request.query.get('id', '0'), 'delay': delay, 'ok': True})

def _run_server():
    global _server_loop, _server_runner
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    _server_loop = loop
    app = web.Application()
    app.router.add_get('/delay', handle_delay)
    runner = web.AppRunner(app)
    loop.run_until_complete(runner.setup())
    site = web.TCPSite(runner, HOST, PORT)
    loop.run_until_complete(site.start())
    _server_runner = runner
    loop.run_forever()

_thread = threading.Thread(target=_run_server, daemon=True)
_thread.start()
time.sleep(0.5)  # esperar a que el server levante

# Sanity check
r = requests.get(f'{BASE}/delay', params={'delay': 0, 'id': 'probe'}, timeout=2)
print(f'server OK: status={r.status_code}, body={r.text[:80]}')

## 2. Baseline sync con `requests` (50 endpoints, delay=0.1 s)

In [ ]:
N = 50
DELAY = 0.1
urls = [f'{BASE}/delay?delay={DELAY}&id={i}' for i in range(N)]

t0 = time.perf_counter()
results_sync = []
for u in urls:
    try:
        r = requests.get(u, timeout=5)
        results_sync.append(r.status_code)
    except Exception as e:
        results_sync.append(f'err:{type(e).__name__}')
t_sync = time.perf_counter() - t0

ok = sum(1 for s in results_sync if s == 200)
print(f'sync requests : {t_sync:.2f}s | {ok}/{N} OK (resto 503 simulado)')
print(f'esperado teórico mínimo: {N * DELAY:.2f}s')

## 3. Async con `httpx.AsyncClient` + `asyncio.gather`

In [ ]:
async def fetch_all_httpx(urls):
    async with httpx.AsyncClient(timeout=5.0) as client:
        tasks = [client.get(u) for u in urls]
        responses = await asyncio.gather(*tasks, return_exceptions=True)
    return [r.status_code if hasattr(r, 'status_code') else f'err:{type(r).__name__}' for r in responses]

t0 = time.perf_counter()
results_async = asyncio.run(fetch_all_httpx(urls))
t_async = time.perf_counter() - t0

ok = sum(1 for s in results_async if s == 200)
print(f'async httpx   : {t_async:.2f}s | {ok}/{N} OK')
print(f'speedup vs sync: {t_sync / t_async:.1f}x')

## 4. Rate limiting con `asyncio.Semaphore`

Limitamos a 10 requests concurrentes — útil para respetar rate limits de APIs externas.

In [ ]:
async def fetch_one(client, sem, url):
    async with sem:
        r = await client.get(url)
        return r.status_code

async def fetch_all_limited(urls, max_concurrent=10):
    sem = asyncio.Semaphore(max_concurrent)
    async with httpx.AsyncClient(timeout=5.0) as client:
        return await asyncio.gather(*[fetch_one(client, sem, u) for u in urls], return_exceptions=True)

for limit in [1, 5, 10, 25, 50]:
    t0 = time.perf_counter()
    asyncio.run(fetch_all_limited(urls, max_concurrent=limit))
    print(f'  Semaphore({limit:3d}): {time.perf_counter() - t0:.2f}s')

## 5. Retry con backoff exponencial (implementación manual)

In [ ]:
async def fetch_with_retry(client, url, max_attempts=5, base=0.2):
    last_exc = None
    for attempt in range(max_attempts):
        try:
            r = await client.get(url)
            if r.status_code < 500:
                return r.status_code, attempt + 1
            last_exc = RuntimeError(f'server {r.status_code}')
        except Exception as e:
            last_exc = e
        wait = base * (2 ** attempt) + random.uniform(0, 0.05)
        await asyncio.sleep(wait)
    return f'fail:{last_exc}', max_attempts

async def fetch_all_retry(urls):
    async with httpx.AsyncClient(timeout=5.0) as client:
        return await asyncio.gather(*[fetch_with_retry(client, u) for u in urls])

t0 = time.perf_counter()
results_retry = asyncio.run(fetch_all_retry(urls))
t_retry = time.perf_counter() - t0

ok = sum(1 for s, _ in results_retry if s == 200)
attempts = [a for _, a in results_retry]
print(f'retry async: {t_retry:.2f}s | {ok}/{N} OK | intentos: min={min(attempts)} max={max(attempts)} avg={sum(attempts)/len(attempts):.2f}')

## 6. Productor-consumidor con `asyncio.Queue`

El productor genera URLs; un pool de N workers las consume. Patrón ideal para pipelines de scraping con backpressure natural.

In [ ]:
async def producer(queue, urls):
    for u in urls:
        await queue.put(u)
    # señal de fin: una por worker
    for _ in range(N_WORKERS):
        await queue.put(None)

async def worker(name, queue, client, out):
    while True:
        url = await queue.get()
        if url is None:
            queue.task_done()
            break
        try:
            r = await client.get(url)
            out.append((name, r.status_code))
        except Exception as e:
            out.append((name, f'err:{type(e).__name__}'))
        finally:
            queue.task_done()

N_WORKERS = 8

async def run_pipeline(urls):
    queue = asyncio.Queue(maxsize=20)
    out = []
    async with httpx.AsyncClient(timeout=5.0) as client:
        workers = [asyncio.create_task(worker(f'w{i}', queue, client, out)) for i in range(N_WORKERS)]
        await producer(queue, urls)
        await asyncio.gather(*workers)
    return out

t0 = time.perf_counter()
pipeline_out = asyncio.run(run_pipeline(urls))
t_pipe = time.perf_counter() - t0

from collections import Counter
by_worker = Counter(w for w, _ in pipeline_out)
ok = sum(1 for _, s in pipeline_out if s == 200)
print(f'pipeline ({N_WORKERS} workers): {t_pipe:.2f}s | {ok}/{N} OK')
print(f'reparto por worker: {dict(by_worker)}')

## 7. `httpx` vs `aiohttp` head-to-head

In [ ]:
async def fetch_all_aiohttp(urls):
    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=5)) as session:
        async def one(u):
            async with session.get(u) as r:
                return r.status
        return await asyncio.gather(*[one(u) for u in urls], return_exceptions=True)

t0 = time.perf_counter(); asyncio.run(fetch_all_httpx(urls));   t_h = time.perf_counter() - t0
t0 = time.perf_counter(); asyncio.run(fetch_all_aiohttp(urls)); t_a = time.perf_counter() - t0

print(f'httpx  : {t_h:.2f}s')
print(f'aiohttp: {t_a:.2f}s')
print('(performance casi idéntica; elegí por DX — httpx tiene API sync+async unificada)')

## 8. Timeouts: comportamiento por request

In [ ]:
# Pedimos un delay de 2s con timeout de 0.3s → debe disparar timeout
async def demo_timeout():
    async with httpx.AsyncClient(timeout=0.3) as client:
        try:
            r = await client.get(f'{BASE}/delay?delay=2&id=slow')
            return ('ok', r.status_code)
        except httpx.TimeoutException as e:
            return ('timeout', str(e))

print(asyncio.run(demo_timeout()))

## 9. Cleanup del mock server

In [ ]:
async def _shutdown():
    if _server_runner:
        await _server_runner.cleanup()

if _server_loop:
    fut = asyncio.run_coroutine_threadsafe(_shutdown(), _server_loop)
    try:
        fut.result(timeout=2)
    except Exception:
        pass
    _server_loop.call_soon_threadsafe(_server_loop.stop)
print('mock server detenido')

## Ejercicios

1. Subí `N` a 500 y comparativa sync vs async — el speedup crece linealmente.
2. Reemplazá el retry manual por `tenacity` (`@retry(stop=stop_after_attempt(5), wait=wait_exponential())`).
3. Agregá un endpoint `/post` al mock server y enviá payloads JSON con `client.post(url, json={...})`.
4. Implementá rate limiting por *segundo* (no solo concurrencia) — pista: `asyncio.sleep` después de cada request del worker.
5. Medí el speedup en función de la concurrencia (Semaphore N=1..100) y dibujá la curva con matplotlib.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y **ejecutables sin internet** de los ejercicios del README. Cada bloque incluye comentarios y comprobaciones (`assert`/`print`). Intenta resolverlos tú antes de mirar.

In [ ]:
import threading, time, json
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import nest_asyncio
nest_asyncio.apply()     # permite asyncio.run() dentro del notebook

class H(BaseHTTPRequestHandler):
    def log_message(self, *a):
        pass
    def do_GET(self):
        time.sleep(0.02)     # simula latencia de red por request
        body = json.dumps({'path': self.path}).encode()
        self.send_response(200); self.send_header('Content-Type', 'application/json')
        self.end_headers(); self.wfile.write(body)

_server = ThreadingHTTPServer(('127.0.0.1', 0), H)
BASE_URL = f'http://127.0.0.1:{_server.server_address[1]}'
threading.Thread(target=_server.serve_forever, daemon=True).start()
N = 30
urls = [f'{BASE_URL}/item/{i}' for i in range(N)]
print('Servidor async-demo en', BASE_URL, '| N =', N, 'URLs')

**Ejercicio 1 — Sync baseline.** `requests.get` en loop; mide el tiempo.

In [ ]:
import requests, time
t0 = time.perf_counter()
res = [requests.get(u).json()['path'] for u in urls]
t_sync = time.perf_counter() - t0
print(f'Sync: {len(res)} requests en {t_sync:.2f}s')
assert len(res) == N

**Ejercicio 2 — Async con httpx.** `asyncio.gather` sobre `AsyncClient`; compara tiempos.

In [ ]:
import httpx, asyncio, time
async def fetch_all(urls):
    async with httpx.AsyncClient() as c:
        rs = await asyncio.gather(*[c.get(u) for u in urls])
        return [r.json()['path'] for r in rs]
t0 = time.perf_counter()
res_async = asyncio.run(fetch_all(urls))
t_async = time.perf_counter() - t0
print(f'Async httpx: {len(res_async)} requests en {t_async:.2f}s')
assert len(res_async) == N
print(f'Speedup frente a sync: ~{t_sync / t_async:.1f}x (las esperas se solapan).')

**Ejercicio 3 — Semaphore.** Limita a 10 concurrentes (cortesía con rate limits).

In [ ]:
async def fetch_limited(urls, limit=10):
    sem = asyncio.Semaphore(limit)
    async with httpx.AsyncClient() as c:
        async def one(u):
            async with sem:              # como máximo 'limit' requests a la vez
                r = await c.get(u); return r.json()['path']
        return await asyncio.gather(*[one(u) for u in urls])
res_lim = asyncio.run(fetch_limited(urls, 10))
print('Con semáforo (máx 10 concurrentes):', len(res_lim), 'requests')
assert len(res_lim) == N
print('El semáforo evita abrir 1000 conexiones de golpe y que te bloqueen por rate limit.')

**Ejercicio 4 — Retry exponencial.** `tenacity` reintenta con espera creciente.

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential
intentos = {'n': 0}
@retry(stop=stop_after_attempt(5), wait=wait_exponential(min=0.01, max=0.1))
def fetch_flaky():
    intentos['n'] += 1
    if intentos['n'] < 3:
        raise ConnectionError('fallo simulado')   # falla 2 veces
    return 'ok'
print('Resultado:', fetch_flaky(), '| intentos:', intentos['n'])
assert intentos['n'] == 3
print('Reintenta con backoff exponencial hasta tener éxito (o agotar los 5 intentos).')

**Ejercicio 5 — httpx vs aiohttp.** Mismo benchmark con ambos clientes.

In [ ]:
import aiohttp, httpx, asyncio, time
async def bench_httpx(urls):
    async with httpx.AsyncClient() as c:
        await asyncio.gather(*[c.get(u) for u in urls])
async def bench_aiohttp(urls):
    async with aiohttp.ClientSession() as s:
        async def one(u):
            async with s.get(u) as r:
                await r.json()
        await asyncio.gather(*[one(u) for u in urls])
t0 = time.perf_counter(); asyncio.run(bench_httpx(urls));   th = time.perf_counter() - t0
t0 = time.perf_counter(); asyncio.run(bench_aiohttp(urls)); ta = time.perf_counter() - t0
print(f'httpx:   {th:.2f}s')
print(f'aiohttp: {ta:.2f}s')
print('Rendimiento similar; httpx ofrece mejor DX (API sync+async, HTTP/2).')